In [1]:
import random
import numpy as np
import torch
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

C:\Users\uqpua\miniforge3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def build_dataset(tokenizer, train_size=100, val_size=20, seed=42):
    def convert_to_prompt(example):
        instruction = example.get("instruction", "")
        context = example.get("context", "") or ""
        response = example.get("response", "")

        if context.strip():
            prompt = (
                "### Instruction: \n" f"{instruction} \n\n"
                "### Context: \n" f"{context} \n\n"
                "### Response: \n"
            )
        else:
            prompt = (
                "### Instruction: \n" f"{instruction} \n\n"
                "### Response: \n"
            )

        text = prompt + response + tokenizer.eos_token
        return {"text": text}

    ds = load_dataset("databricks/databricks-dolly-15k")["train"]
    ds = ds.shuffle(seed=seed)
    
    val_ds = ds.select(range(val_size))
    train_ds = ds.select(range(val_size, val_size + train_size))

    train_prom = train_ds.map(convert_to_prompt, remove_columns=train_ds.column_names)
    val_prom = val_ds.map(convert_to_prompt, remove_columns=val_ds.column_names)

    return train_prom, val_prom
    

def train_fn(model_name="distilgpt2", output_dir="./result"):
    set_seed(42)
    device = torch.device("cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.to(device)

    train, val = build_dataset(tokenizer, train_size=10, val_size=2, seed=42)
    # print(train)
    # print(train[0])
    # print(train[0]["text"])

    # args = SFTConfig(
    #     output_dir=output_dir,
    #     eval_strategy="epoch",
    #     num_train_epochs=1,
    #     per_device_train_batch_size=2,
    #     per_device_eval_batch_size=2,
    #     learning_rate=5e-5,
    #     save_strategy="no",
    #     logging_strategy="steps",
    #     logging_steps=10,
    #     use_cpu=True,
    #     dataset_text_field="text", # 기본값이 'text'
    #     max_length=256,
    #     # load_best_model_at_end=True
    # )

    args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="epoch",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=5e-5,
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=10,
        no_cuda=True,
        fp16=False,
        bf16=False,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train,
        eval_dataset=val,
        processing_class=tokenizer
    )

    trainer.train()
    metrics = trainer.evaluate()
    # print("Eval metrics:", metrics)

    return metrics
    

SyntaxError: parameter without a default follows parameter with a default (579077859.py, line 6)

In [ ]:
result = train_fn()

Truncating eval dataset: 100%|████████████████████████████████████████████████████| 2/2 [00:00<00:00, 51.93 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss


In [ ]:
# train_dpo.py
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct")
train_dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")

training_args = DPOConfig(output_dir="Qwen2-0.5B-DPO")
trainer = DPOTrainer(model=model, args=training_args, processing_class=tokenizer, train_dataset=train_dataset)
trainer.train()